In [0]:
data = [('Alice', 'Badminton,Tennis'), ('Bob', 'Tennis,Cricket'), ('Julie', 'Cricket,Carroms')]
columns = ['Name', 'Hobbies']
df = spark.createDataFrame(data, columns)
display(df)

In [0]:
from pyspark.sql.functions import split, explode
df.select(df.Name, explode(split(df.Hobbies, ",")).alias("hobbies")).display()

In [0]:
data = [('Goa', '', 'AP'), ('', 'AP', None), (None, '', 'bglr')]
columns = ['City1', 'City2', 'City3']
df = spark.createDataFrame(data, columns)
display(df)

In [0]:
from pyspark.sql.functions import *
df=df.withColumn('firstnotnull', coalesce(df.City1, df.City2, df.City3))
display(df)

In [0]:
data1 = [('Goa', None, 'AP'), (None, 'AP', None), (None, None, 'bglr')]
columns = ['City1', 'City2', 'City3']
df1 = spark.createDataFrame(data1, columns)
display(df1)
df1=df1.withColumn('firstnotnull', coalesce(df1.City1, df1.City2, df1.City3))
display(df1)


In [0]:
data1 = [('Goa', '', 'AP'), ('', 'AP', None), (None, '', 'bglr')]
columns = ['City1', 'City2', 'City3']
df1 = spark.createDataFrame(data1, columns)
display(df1)
df1 = df1.withColumn('firstnotnull', coalesce(
    when(df1.City1=='', None).otherwise(df1.City1), 
    when(df1.City2=='', None).otherwise(df1.City2),
    when(df1.City3=='', None).otherwise(df1.City3)))
display(df1)

In [0]:
data1 = [(1, 'Steve'), (2, 'David'), (3, 'John'), (4, 'Shree'), (5, 'Helen')]
data2 = [(1,'SQL',90), (1,'PySpark',100), (2,'SQL',70), (2,'PySpark',60), (3,'SQL',30), (3,'PySpark',20), (4,'SQL',50), (4,'PySpark',50), (5,'SQL',45), (5,'PySpark',45)]

schema1 = ['ID', 'Name']
schema2 = ['ID', 'Subject', 'Mark']

df1 = spark.createDataFrame(data1, schema1)
df2 = spark.createDataFrame(data2, schema2)

display(df1)
display(df2)

In [0]:
df_join = df1.join(df2, df1.ID==df2.ID).drop(df2.ID)
display(df_join)

In [0]:
df_per = df_join.groupBy('ID', 'Name').agg((sum("Mark")/count('*')).alias('Percentage'))
df_per.display()

In [0]:
df_per.select("*", (when(df_per.Percentage>=70, 'Distinction')
              .when((df_per.Percentage<70) & (df_per.Percentage>=60), 'First Class')
              .when((df_per.Percentage<60) & (df_per.Percentage>=50), 'Second Class')
              .when((df_per.Percentage<50) & (df_per.Percentage>=40), 'Third Class')
              .when(df_per.Percentage<40, 'Fail')).alias('class')).display()

In [0]:
data1 = [(1,'A', 1000, 'IT'), (2,'B', 1500, 'IT'), (3,'C', 2500, 'HR'), (4,'D', 3000, 'HR'), (5,'E', 2000, 'HR'), (6,'F', 1000, 'HR'), (7,'G', 4000, 'Sales'), (8,'H', 4000, 'Sales'), (9,'I', 1000, 'Sales'), (10,'J',2000, 'Sales')]
schema1 = ['EmpId', 'EmpName', 'Salary', 'DeptName']
df = spark.createDataFrame(data1, schema1)
display(df)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import *

df_rank = df.select('*', dense_rank().over(Window.partitionBy(df.DeptName).orderBy(df.Salary.desc())).alias('rank'))
df_rank.display()

In [0]:
# employees salary info
data1 = [(100, 'Raj', None, 1, '01-04-23', 50000), (200, 'Joanne', 100,1, '01-04-23', 4000), (200, 'Joanne', 100,1, '13-04-23', 4500), (200, 'Joanne', 100, 1, '14-04-23', 4020)]

schema1 = ['EmpId', 'EmpName', 'mgrid', 'deptid', 'salarydt', 'salary']
df_salary = spark.createDataFrame(data1, schema1)
display(df_salary)

# department dataframe
data2 = [(1, 'IT'), (2, 'HR')]
schema2 = ['deptid', 'deptname']
df_dept = spark.createDataFrame(data2, schema2)
display(df_dept)

In [0]:
from pyspark.sql.functions import *

In [0]:
df = df_salary.withColumn('Newsaldt', to_date('salarydt', 'dd-MM-yy'))
display(df)

In [0]:
df1 = df.join(df_dept, ['deptid'])
display(df1)

In [0]:
df2 = df1.alias('a').join(df1.alias('b'), col('a.mgrid') == col('b.EmpId'), 'left').select(col('a.deptname'), col('b.EmpName').alias('ManagerName'), col('a.EmpName'), col('a.Newsaldt'), col('a.salary'))
df2.display()

In [0]:
df2.groupBy('deptname', 'ManagerName', 'EmpName', year('Newsaldt').alias('Year'), month('Newsaldt').alias('Month')).sum('salary').display()